# Predicting Song Hits from Audio Characteristics
### DSC 148 — Course Project

**Research question:** *What audio characteristics make a song likely to become a hit?*

We frame this as a **binary classification** problem on the Spotify 1M Tracks dataset.
A track is labeled a **hit** if its `popularity` is in the top 10% of all tracks.

We compare four models:
| Role | Model | Rationale |
|---|---|---|
| Baseline 1 | Majority Class Predictor | Minimum performance bar |
| Baseline 2 | Logistic Regression | Simple, standard linear model |
| Model 1 | Random Forest | Nonlinear, feature interactions |
| Model 2 | XGBoost | State-of-the-art on tabular data |

---

## 0. Setup

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             precision_score, recall_score, confusion_matrix,
                             roc_curve, precision_recall_curve, average_precision_score,
                             classification_report)
import xgboost as xgb

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
RNG = 42
os.makedirs("figures", exist_ok=True)
results = {}   # collect metrics for the report

## 1. Dataset (5%)

The [Spotify 1 Million Tracks](https://www.kaggle.com/datasets/amitanshjoshi/spotify-1million-tracks)
dataset contains ~1.16M tracks with audio features computed by Spotify
(danceability, energy, valence, etc.), plus a `popularity` score (0–100).

> **Run-once cell.** If you already have the dataset cached locally, set `DATA_PATH`
> in the next cell to that file and skip this download.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("amitanshjoshi/spotify-1million-tracks")
print("Path to dataset files:", path)
print(os.listdir(path))

In [ ]:
# Point this at the CSV. By default uses the kagglehub cache from the cell above.
DATA_PATH = os.path.join(path, "spotify_data.csv")

df = pd.read_csv(DATA_PATH)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
print("Raw shape:", df.shape)
df.head()

### 1.1 Cleaning

In [ ]:
AUDIO_FEATURES = ["danceability", "energy", "key", "loudness", "mode",
                  "speechiness", "acousticness", "instrumentalness",
                  "liveness", "valence", "tempo", "duration_ms", "time_signature"]

print("Duplicate track_ids:", df.duplicated(subset=["track_id"]).sum())
df = df.drop_duplicates(subset=["track_id"]).reset_index(drop=True)
df = df.dropna(subset=AUDIO_FEATURES + ["popularity"]).reset_index(drop=True)
print("Clean shape:", df.shape)
df[AUDIO_FEATURES + ["popularity", "year"]].describe().round(3)

### 1.2 Exploratory Data Analysis

#### Defining a "hit"
There is no ready-made hit label, so we derive one from `popularity`.
Most tracks have low popularity, so we treat the **top 10%** (90th percentile)
as hits. This gives a clear, imbalanced classification target that reflects the
real-world rarity of hit songs.

In [ ]:
HIT_PERCENTILE = 90
thr = np.percentile(df["popularity"], HIT_PERCENTILE)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df["popularity"], bins=50, color="#4C72B0", edgecolor="white")
ax.axvline(thr, color="#C44E52", lw=2.5, ls="--",
           label=f"Hit threshold (P{HIT_PERCENTILE}) = {thr:.0f}")
ax.set_xlabel("Popularity (0-100)"); ax.set_ylabel("Track count")
ax.set_title("Distribution of Track Popularity"); ax.legend()
fig.tight_layout(); fig.savefig("figures/01_popularity_dist.png", dpi=150); plt.show()

df["is_hit"] = (df["popularity"] >= thr).astype(int)
hit_rate = df["is_hit"].mean()
print(f"threshold={thr:.1f}  hit_rate={hit_rate:.3%}  "
      f"#hits={df['is_hit'].sum():,}  #non-hits={(len(df)-df['is_hit'].sum()):,}")
results.update(hit_threshold=float(thr), hit_rate=float(hit_rate), n_total=int(len(df)))

#### Correlation structure

In [ ]:
corr = df[AUDIO_FEATURES + ["popularity"]].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label": "Pearson r"})
ax.set_title("Feature Correlation Matrix")
fig.tight_layout(); fig.savefig("figures/02_corr_heatmap.png", dpi=150); plt.show()

corr["popularity"].drop("popularity").sort_values(key=abs, ascending=False).round(3)

#### How do hits differ from non-hits?

In [ ]:
compare_feats = ["danceability", "energy", "valence", "loudness",
                 "acousticness", "instrumentalness", "speechiness", "tempo"]
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, feat in zip(axes.ravel(), compare_feats):
    for label, color in [(0, "#55A868"), (1, "#C44E52")]:
        sns.kdeplot(df.loc[df["is_hit"] == label, feat], ax=ax, fill=True,
                    alpha=0.4, color=color, label="Hit" if label else "Non-hit")
    ax.set_title(feat); ax.set_xlabel("")
axes.ravel()[0].legend()
fig.suptitle("Audio Feature Distributions: Hits vs Non-hits", y=1.01)
fig.tight_layout(); fig.savefig("figures/03_hit_vs_nonhit.png", dpi=150); plt.show()

In [ ]:
group_means = df.groupby("is_hit")[compare_feats].mean().T
group_means.columns = ["non_hit_mean", "hit_mean"]
group_means["abs_diff"] = (group_means["hit_mean"] - group_means["non_hit_mean"]).abs()
group_means = group_means.sort_values("abs_diff", ascending=False)
group_means.to_csv("figures/table_feature_means.csv")
group_means.round(3)

#### Popularity over time

In [ ]:
year_pop = df.groupby("year")["popularity"].mean()
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(year_pop.index, year_pop.values, marker="o", color="#4C72B0")
ax.set_xlabel("Year"); ax.set_ylabel("Mean popularity")
ax.set_title("Mean Track Popularity by Release Year")
fig.tight_layout(); fig.savefig("figures/04_popularity_by_year.png", dpi=150); plt.show()

## 2. Predictive Task (5%)

**Task:** Given a track's audio characteristics, predict whether it is a hit
(`is_hit = 1`, top 10% popularity) or not.

**Evaluation.** Because the classes are imbalanced (~10% positives), accuracy alone
is misleading — a model that always predicts "non-hit" scores ~90%. We therefore
report **F1, ROC-AUC, and PR-AUC** alongside accuracy, and treat **F1 / PR-AUC** as
the primary metrics.

**Baselines.**
- *Majority Class Predictor* — always predicts non-hit; defines the floor.
- *Logistic Regression* — assumes hit probability is a linear function of (scaled) features.

**Why trees can win.** Audio features interact nonlinearly (e.g. high energy helps only
when danceability is also high). Logistic regression cannot capture such interactions;
Random Forest and XGBoost can.

**Feature set.** All 13 audio characteristics plus `year` as release-era context.
We deliberately exclude `artist_name`/`genre` so the model answers the *audio* question.

In [ ]:
FEATURES = AUDIO_FEATURES + ["year"]
X = df[FEATURES].copy()
y = df["is_hit"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RNG)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train hit rate:", y_train.mean().round(4), " Test hit rate:", y_test.mean().round(4))

In [ ]:
def best_threshold(y_true, y_prob):
    """Pick the probability cutoff that maximizes F1 (better under imbalance)."""
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    return float(thr[np.argmax(f1[:-1])])

def evaluate(name, y_true, y_pred, y_prob=None):
    m = {"accuracy": accuracy_score(y_true, y_pred),
         "precision": precision_score(y_true, y_pred, zero_division=0),
         "recall": recall_score(y_true, y_pred, zero_division=0),
         "f1": f1_score(y_true, y_pred, zero_division=0)}
    if y_prob is not None:
        m["roc_auc"] = roc_auc_score(y_true, y_prob)
        m["pr_auc"] = average_precision_score(y_true, y_prob)
    results.setdefault("models", {})[name] = m
    print(f"[{name}]  " + "  ".join(f"{k}={v:.4f}" for k, v in m.items()))
    return m

## 3. Model (5%)

We train the two baselines and two tree ensembles below. For the tree models we also
tune the decision threshold on the training set (via the PR curve) instead of using the
default 0.5, since the positive class is rare. `class_weight="balanced"` /
`scale_pos_weight` further counteract imbalance.

### 3.1 Baseline 1 — Majority Class Predictor

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
evaluate("Majority Class", y_test, dummy.predict(X_test),
         dummy.predict_proba(X_test)[:, 1]);

### 3.2 Baseline 2 — Logistic Regression

In [ ]:
logit = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RNG)),
]).fit(X_train, y_train)
lprob = logit.predict_proba(X_test)[:, 1]
evaluate("Logistic Regression", y_test, logit.predict(X_test), lprob);

### 3.3 Model 1 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                            class_weight="balanced", n_jobs=-1, random_state=RNG)
rf.fit(X_train, y_train)
rprob = rf.predict_proba(X_test)[:, 1]
rf_t = best_threshold(y_train, rf.predict_proba(X_train)[:, 1])
print("RF tuned threshold:", round(rf_t, 3))
evaluate("Random Forest", y_test, (rprob >= rf_t).astype(int), rprob);

### 3.4 Model 2 — XGBoost

In [ ]:
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
xgb_clf = xgb.XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos,
    eval_metric="logloss", tree_method="hist", n_jobs=-1, random_state=RNG)
xgb_clf.fit(X_train, y_train)
xprob = xgb_clf.predict_proba(X_test)[:, 1]
xgb_t = best_threshold(y_train, xgb_clf.predict_proba(X_train)[:, 1])
print("XGB tuned threshold:", round(xgb_t, 3))
xp = (xprob >= xgb_t).astype(int)
evaluate("XGBoost", y_test, xp, xprob);

## 4. Literature (5%)

*To be written in the report.* Points to cover:
- Hit Song Science (HSS): Pachet & Roy (2008) argued popularity is hard to predict
  from audio alone; later work (e.g. Interiano et al. 2018, Yang et al. 2017) showed
  partial predictability using audio + temporal features.
- The Spotify audio features here are the same ones used in many of these studies.
- How our top features and accuracy compare to published HSS results.
- Our novelty: large (~1M track) corpus, modern gradient boosting, threshold tuning
  for the rare-hit setting.

## 5. Results (5%)

### 5.1 Model comparison

In [ ]:
res_df = pd.DataFrame(results["models"]).T[
    ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]]
res_df.to_csv("figures/table_model_comparison.csv")
res_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
res_df[["accuracy", "f1", "roc_auc"]].plot.bar(
    ax=ax, width=0.8, color=["#4C72B0", "#C44E52", "#55A868"])
ax.set_ylabel("Score"); ax.set_ylim(0, 1.0); ax.set_title("Model Performance Comparison")
ax.set_xticklabels(res_df.index, rotation=20, ha="right"); ax.legend(loc="lower right")
fig.tight_layout(); fig.savefig("figures/05_model_comparison.png", dpi=150); plt.show()

### 5.2 ROC and Precision-Recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for name, prob in [("Logistic Regression", lprob), ("Random Forest", rprob), ("XGBoost", xprob)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC={roc_auc_score(y_test, prob):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random")
axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curves")
axes[0].legend(loc="lower right")

for name, prob in [("Logistic Regression", lprob), ("Random Forest", rprob), ("XGBoost", xprob)]:
    prec, rec, _ = precision_recall_curve(y_test, prob)
    axes[1].plot(rec, prec, lw=2, label=f"{name} (AP={average_precision_score(y_test, prob):.3f})")
axes[1].axhline(y_test.mean(), color="k", ls="--", alpha=0.5, label=f"Baseline ({y_test.mean():.3f})")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curves")
axes[1].legend(loc="upper right")
fig.tight_layout()
fig.savefig("figures/06_roc_curves.png", dpi=150)
fig.savefig("figures/07_pr_curves.png", dpi=150); plt.show()

### 5.3 Confusion matrix (best model by F1)

In [ ]:
best_name = res_df["f1"].idxmax()
best_pred = {"Random Forest": (rprob >= rf_t).astype(int),
             "XGBoost": xp, "Logistic Regression": logit.predict(X_test)}[best_name]
results["best_model"] = best_name
print("Best model by F1:", best_name)
print(classification_report(y_test, best_pred, target_names=["Non-hit", "Hit"]))

cm = confusion_matrix(y_test, best_pred)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Non-hit", "Hit"], yticklabels=["Non-hit", "Hit"])
ax.set(xlabel="Predicted", ylabel="Actual", title=f"Confusion Matrix — {best_name}")
fig.tight_layout(); fig.savefig("figures/08_confusion_best.png", dpi=150); plt.show()

### 5.4 Which audio characteristics drive hits?

This is the direct answer to the research question.

In [ ]:
imp = pd.DataFrame({
    "feature": FEATURES,
    "rf_importance": rf.feature_importances_,
    "xgb_importance": xgb_clf.feature_importances_,
}).sort_values("xgb_importance", ascending=False)
imp.to_csv("figures/table_feature_importance.csv", index=False)

order = imp.sort_values("rf_importance")
y_pos = np.arange(len(order))
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(y_pos - 0.2, order["rf_importance"], height=0.4, label="Random Forest", color="#4C72B0")
ax.barh(y_pos + 0.2, order["xgb_importance"], height=0.4, label="XGBoost", color="#C44E52")
ax.set_yticks(y_pos); ax.set_yticklabels(order["feature"])
ax.set(xlabel="Importance", title="Feature Importance by Audio Characteristic"); ax.legend()
fig.tight_layout(); fig.savefig("figures/09_feature_importance.png", dpi=150); plt.show()
imp.round(4)

### 5.5 Ablation study

We drop the three most important features and retrain XGBoost. A large F1 drop
confirms those audio characteristics carry most of the predictive signal.

In [ ]:
ablation = {"full": f1_score(y_test, xp)}
top3 = imp["feature"].head(3).tolist()
xgb_ab = xgb.XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, scale_pos_weight=scale_pos, eval_metric="logloss",
    tree_method="hist", n_jobs=-1, random_state=RNG)
xgb_ab.fit(X_train.drop(columns=top3), y_train)
ab_prob = xgb_ab.predict_proba(X_test.drop(columns=top3))[:, 1]
ab_t = best_threshold(y_train, xgb_ab.predict_proba(X_train.drop(columns=top3))[:, 1])
ablation[f"drop_top3 ({', '.join(top3)})"] = f1_score(y_test, (ab_prob >= ab_t).astype(int))
results["ablation"] = {k: float(v) for k, v in ablation.items()}
pd.Series(ablation).round(4)

### 5.6 Save all metrics

In [ ]:
with open("figures/metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved figures, tables, and metrics.json to ./figures/")
print("Best model by F1:", results["best_model"])

---
### Takeaways (fill in after running on the full dataset)
- Both tree ensembles beat the majority-class floor and logistic regression on F1/PR-AUC.
- The most predictive audio characteristics are **{top features from §5.4}**.
- Predicting hits from audio alone remains hard (moderate PR-AUC), consistent with the
  Hit Song Science literature — audio explains part, but not all, of popularity.